# Lab08.2 — BNN-PYNQ pretrained traffic-sign recognition on Zybo ARM

**Platform:** Digilent Zybo Legacy (Zynq-7010), PYNQ 3.0.1.
**Model:** original trained CNV-W1A1 (`road-signs`), 43 GTSRB signs + one `Not a roadsign` class.
**Execution:** ARM software library from [`onigas/BNN-PYNQ`](https://github.com/onigas/BNN-PYNQ). **No FPGA bitstream is loaded.**

**Verified on Zybo:** five distinct official GTSRB test images were classified correctly (5/5), mean library-reported ARM time 1.587 s/image. This is illustrative accuracy on five selected samples, NOT the full-dataset accuracy. Choose one with `IMAGE_INDEX`; optionally run the five-image test.

**Requirements:** the original GTSRB test CSV and five genuine `.ppm` photographs, plus the pinned original ARM `.so` and pretrained `road-signs` parameters, stored under `Lab08/`. These assets are intended to be distributed with the laboratory. No network connection, QONNX, FINN, PyTorch or Vivado is required for normal execution.

**Safety:** the C++ library is called only in a child process. It may crash without terminating the Jupyter kernel. This test does not reprogram programmable logic.


## 1 — Check Zybo environment and locate the real photograph

Run this first and send the printed diagnostics if any path is missing. All paths are relative to `Lab08/notebooks/`.


In [ ]:
from pathlib import Path
import csv
import os
import platform
import shutil
import sys
import numpy as np
from PIL import Image

ROOT = Path('..').resolve()
IMAGE = ROOT / 'data/gtsrb/GTSRB/Final_Test/Images/00105.ppm'
GT_CSV = ROOT / 'data/gtsrb/GT-final_test.csv'
ASSET_DIR = ROOT / 'assets/bnn_pynq_sw'
PARAMS = ASSET_DIR / 'params'
SO_FILE = ASSET_DIR / 'python_sw-cnvW1A1-pynqZ1-Z2.so'
CIFAR_FILE = ASSET_DIR / 'test_00105.cifar'

print('Python:', sys.version.split()[0])
print('Architecture:', platform.machine())
print('Platform:', platform.platform())
print('Lab08 root:', ROOT)
print('Image exists:', IMAGE.is_file(), IMAGE)
print('Ground-truth CSV exists:', GT_CSV.is_file())
print('Software .so exists:', SO_FILE.is_file())
print('Parameter files present:', len(list(PARAMS.glob('*.bin'))) if PARAMS.exists() else 0)
assert platform.machine().startswith('arm'), 'This notebook is intended to test the ARM software library on Zybo.'
assert IMAGE.is_file(), 'Copy the verified Lab08.1 photograph 00105.ppm into the expected data directory.'


## 2 — Use the ORIGINAL pretrained library and parameters

The 32-bit ARM library and 179 parameter/label files come from the pinned BNN-PYNQ commit `d39c1c75403db571f54dd0f0cf27ae5104ac968e`. Keep the original binary files and network weights unchanged.

**Normal laboratory run:** `DOWNLOAD_ASSETS = False` in the next cell. The library and parameters should already be available in `Lab08/assets/bnn_pynq_sw/`. Continue to Section 3 to check their completeness.

**Recovery only:** if these files are absent, set `DOWNLOAD_ASSETS = True` to fetch and verify the original files from the pinned GitHub source (board Internet required), or extract a previously prepared `bnn_pynq_sw.zip` using the optional offline cell. The notebook does not generate substitute network weights.


In [ ]:
# Normal laboratory run uses the original assets already distributed in Lab08/assets/.
# Set True only to recover missing files from the pinned upstream source.
DOWNLOAD_ASSETS = False

if DOWNLOAD_ASSETS:
    import concurrent.futures
    import hashlib
    import json
    import time
    import urllib.error
    import urllib.request

    COMMIT = 'd39c1c75403db571f54dd0f0cf27ae5104ac968e'
    BASE = 'https://raw.githubusercontent.com/onigas/BNN-PYNQ/' + COMMIT + '/'
    DIRECTORY = 'bnn/params/road-signs/cnvW1A1'
    LIST_URL = ('https://api.github.com/repos/onigas/BNN-PYNQ/contents/'
                + DIRECTORY + '?ref=' + COMMIT)
    def get_bytes(url, tries=3):
        for trial in range(tries):
            try:
                req = urllib.request.Request(url, headers={'User-Agent':'Lab08-Zybo-education'})
                with urllib.request.urlopen(req, timeout=35) as response:
                    return response.read()
            except Exception:
                if trial == tries - 1:
                    raise
                time.sleep(1 + trial)

    def git_sha(data):
        return hashlib.sha1(b'blob ' + str(len(data)).encode('ascii') + b'\0' + data).hexdigest()

    try:
        manifest = json.loads(get_bytes(LIST_URL).decode('utf-8'))
        if not isinstance(manifest, list) or len(manifest) != 179:
            raise RuntimeError('Unexpected pretrained parameter listing')
        jobs = [(x['path'], PARAMS / x['name'], x['size'], x['sha'])
                for x in manifest if x['type']=='file']
        jobs.append(('bnn/libraries/pynqZ1-Z2/python_sw-cnvW1A1-pynqZ1-Z2.so',
                     SO_FILE, 3697920, '564ca83bf64c9476af789d263f05e5106d33e54a'))

        def download_one(job):
            src, dest, size, sha = job
            dest.parent.mkdir(parents=True, exist_ok=True)
            if dest.is_file() and dest.stat().st_size == size:
                existing = dest.read_bytes()
                if git_sha(existing) == sha:
                    return 'cached'
            data = get_bytes(BASE + src)
            if len(data) != size or git_sha(data) != sha:
                raise ValueError('Downloaded file checksum mismatch: ' + src)
            temp_path = dest.with_suffix(dest.suffix + '.part')
            temp_path.write_bytes(data)
            os.replace(str(temp_path), str(dest))
            return 'downloaded'

        with concurrent.futures.ThreadPoolExecutor(max_workers=5) as pool:
            states = list(pool.map(download_one, jobs))
        print('Assets verified:', len(states), '| downloaded:', states.count('downloaded'),
              '| already present:', states.count('cached'))
        print('Library:', SO_FILE)
    except Exception as e:
        print('Download unavailable or incomplete:', type(e).__name__, str(e))
        print('If the Zybo has no Internet, use the manual ZIP transfer described above,')
        print('set DOWNLOAD_ASSETS=False and continue at section 3.')


### Optional — Extract the offline asset ZIP

Only needed if you uploaded `Lab08/assets/bnn_pynq_sw.zip` from Windows. It must contain the top-level `bnn_pynq_sw/` folder with the `.so` file and `params/` subfolder. The extraction checks the paths before writing files.


In [ ]:
import zipfile

TRANSFER_ZIP = ROOT / 'assets' / 'bnn_pynq_sw.zip'
if TRANSFER_ZIP.is_file():
    with zipfile.ZipFile(str(TRANSFER_ZIP)) as zf:
        members = zf.namelist()
        if not all(m.startswith('bnn_pynq_sw/') and '..' not in Path(m).parts
                   for m in members):
            raise ValueError('Unexpected ZIP layout: expected bnn_pynq_sw/...')
        zf.extractall(str(ROOT / 'assets'))
    print('Offline assets extracted from:', TRANSFER_ZIP)
else:
    print('No offline ZIP found; normal if assets were downloaded directly.')


## 3 — Asset and shared-library compatibility checks

Check the exact number of parameter files and inspect the dynamic dependencies. **Do not load the PYNQ-Z2 bitstream.** A missing dependency in `ldd` may explain a later loading failure. The `.so` check does not by itself prove inference will work.


In [ ]:
import subprocess

files = list(PARAMS.glob('*.bin')) if PARAMS.exists() else []
classes_path = PARAMS / 'classes.txt'
print('Library found:', SO_FILE.is_file())
print('Parameters:', len(files), 'binary files')
print('Classes file:', classes_path.is_file())
if classes_path.is_file():
    CLASS_NAMES = [x.strip() for x in classes_path.read_text(encoding='utf-8').splitlines() if x.strip()]
    print('Class count:', len(CLASS_NAMES))
    print('Class 4:', CLASS_NAMES[4] if len(CLASS_NAMES)>4 else 'MISSING')
    print('Class 43:', CLASS_NAMES[43] if len(CLASS_NAMES)>43 else 'MISSING')
else:
    CLASS_NAMES = []

if SO_FILE.is_file():
    with SO_FILE.open('rb') as f:
        elf_header = f.read(20)
    print('ELF format:', elf_header[:4] == b'\x7fELF')
    print('ELF class:', '32-bit' if elf_header[4]==1 else 'other')
    print('ELF machine:', int.from_bytes(elf_header[18:20], 'little'), '(40 = ARM)')
    if shutil.which('ldd'):
        check = subprocess.run(['ldd', str(SO_FILE)], stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, universal_newlines=True,
                               timeout=20)
        print('ldd output:\n' + check.stdout[:8000])
else:
    print('STOP: upload/download the pretrained software .so before section 5.')

READY = SO_FILE.is_file() and len(files)==178 and len(CLASS_NAMES)==44
print('Assets complete:', READY)


## 4 — Prepare five real, labeled GTSRB examples

This section selects five official test photographs, one per class, using the ground-truth CSV. The five included files are `00105.ppm`, `00093.ppm`, `00001.ppm`, `00008.ppm` and `00003.ppm`, corresponding to the selected classes 4, 14, 1, 25 and 33.

**Normal laboratory run:** leave `DOWNLOAD_MISSING_IMAGES = False`. No Internet is required, and a missing file produces an explicit error instead of an 84 MB download.

**Recovery only:** to restore missing photographs, set this option to `True`; the code can extract just the required examples from the official GTSRB image archive, downloading it if necessary.


In [ ]:
from io import BytesIO
import tempfile
import urllib.request
import zipfile
import matplotlib.pyplot as plt

IMAGE_DIR = ROOT / 'data/gtsrb/GTSRB/Final_Test/Images'
ARCHIVE_PATH = ROOT / 'assets/GTSRB_Final_Test_Images.zip'
DOWNLOAD_MISSING_IMAGES = False  # True only for explicit recovery of missing official PPM images

# Select distinct classes (GTSRB official class IDs).
TARGET_CLASSES = [4, 14, 1, 25, 33]
PREFERRED_IMAGES = {4: '00105.ppm'}

with GT_CSV.open('r', encoding='utf-8-sig', newline='') as csv_file:
    ground_truth = list(csv.DictReader(csv_file, delimiter=';', skipinitialspace=True))

SAMPLES = []
for class_id in TARGET_CLASSES:
    matching = [r for r in ground_truth if int(r['ClassId']) == class_id]
    if not matching:
        raise ValueError('Class {} missing from GT-final_test.csv'.format(class_id))
    requested = PREFERRED_IMAGES.get(class_id)
    if requested is not None:
        matching = [r for r in matching if Path(r['Filename']).name == requested]
        if not matching:
            raise ValueError('{} does not have expected class {}'.format(requested, class_id))
    # Prefer an image already on the board when possible.
    entry = next((r for r in matching if (IMAGE_DIR / Path(r['Filename']).name).is_file()), matching[0])
    name = Path(entry['Filename']).name
    SAMPLES.append({'class_id': class_id, 'name': name, 'path': IMAGE_DIR / name})

missing = [s for s in SAMPLES if not s['path'].is_file()]
print('Selected official test examples:')
for i, s in enumerate(SAMPLES):
    print(' {}: {:>9} | class {:2d}: {:25s} | {}'.format(
        i, s['name'], s['class_id'], CLASS_NAMES[s['class_id']],
        'present' if s['path'].is_file() else 'MISSING'))

if missing and DOWNLOAD_MISSING_IMAGES:
    if ARCHIVE_PATH.is_file():
        archive_source = ARCHIVE_PATH
        print('Using offline GTSRB archive:', archive_source)
        temporary = False
    else:
        import shutil
        free = shutil.disk_usage(str(ASSET_DIR)).free
        if free < 140 * 1024 * 1024:
            raise RuntimeError('Not enough temporary SD space for the 84 MB GTSRB ZIP (need ~140 MB free).')
        archive_source = ASSET_DIR / 'GTSRB_Final_Test_Images.zip.part'
        temporary = True
        url = ('https://sid.erda.dk/public/archives/'
               'daaeac0d7ce1152aea9b61d9f1e19370/GTSRB_Final_Test_Images.zip')
        print('Downloading official GTSRB test archive (~84 MB). One-time operation...')
        try:
            req = urllib.request.Request(url, headers={'User-Agent': 'PYNQ-Zybo-Lab08/1.0'})
            with urllib.request.urlopen(req, timeout=180) as response, archive_source.open('wb') as target:
                total = 0
                next_progress = 10 * 1024 * 1024
                while True:
                    chunk = response.read(1024 * 1024)
                    if not chunk:
                        break
                    target.write(chunk)
                    total += len(chunk)
                    if total >= next_progress:
                        print('Downloaded {:.0f} MB'.format(total / 1048576))
                        next_progress += 10 * 1024 * 1024
            print('Archive downloaded:', round(total / 1048576, 1), 'MB')
        except Exception:
            archive_source.unlink(missing_ok=True)
            raise
    try:
        with zipfile.ZipFile(str(archive_source)) as archive:
            lookup = {}
            for item in archive.infolist():
                if not item.is_dir():
                    normalized = item.filename.replace('\\', '/')
                    if normalized.startswith('GTSRB/Final_Test/Images/'):
                        lookup[Path(normalized).name] = item
            IMAGE_DIR.mkdir(parents=True, exist_ok=True)
            for sample in missing:
                member = lookup.get(sample['name'])
                if member is None:
                    raise FileNotFoundError('{} not found in the official GTSRB archive'.format(sample['name']))
                if member.file_size > 2_000_000:
                    raise ValueError('Unexpectedly large PPM: ' + sample['name'])
                payload = archive.read(member)  # validates the ZIP entry CRC
                with Image.open(BytesIO(payload)) as check:
                    check.verify()
                sample['path'].write_bytes(payload)
                print('Extracted:', sample['name'], len(payload), 'bytes')
    finally:
        if temporary:
            archive_source.unlink(missing_ok=True)

missing = [s['name'] for s in SAMPLES if not s['path'].is_file()]
if missing:
    raise FileNotFoundError('Missing GTSRB images: {}. Copy the named PPM files '
                            'from the official archive to {} and rerun this cell.'.format(missing, IMAGE_DIR))

fig, axes = plt.subplots(1, 5, figsize=(14, 3))
for idx, (ax, s) in enumerate(zip(axes, SAMPLES)):
    with Image.open(str(s['path'])) as img:
        ax.imshow(img.convert('RGB'))
    ax.set_title('{}: {}\n{}'.format(idx, s['name'], CLASS_NAMES[s['class_id']]), fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.show()
print('READY: five distinct GTSRB test images, each with its real CSV label.')


## 5 — Select an image in Python and prepare the CIFAR record

Change **only `IMAGE_INDEX`** to choose one of the five displayed photographs (`0`, `1`, `2`, `3`, `4`). Re-execute this cell **and then section 6**. Each selection uses the identical preprocessing and the same original trained weights. The single-image record is always 3073 bytes.


In [ ]:
# >>> CHANGE THIS NUMBER to 0, 1, 2, 3, or 4 <<<
IMAGE_INDEX = 0

if not isinstance(IMAGE_INDEX, int) or not 0 <= IMAGE_INDEX < len(SAMPLES):
    raise ValueError('IMAGE_INDEX must be an integer from 0 to 4')

SELECTED = SAMPLES[IMAGE_INDEX]
IMAGE = SELECTED['path']
EXPECTED = SELECTED['class_id']
CIFAR_FILE = ASSET_DIR / ('test_' + IMAGE.stem + '.cifar')

def prepare_bnn_cifar(image_path, record_path):
    with Image.open(str(image_path)) as original:
        rgb = original.convert('RGB')
    thumbnail = rgb.copy()
    resample = Image.Resampling.LANCZOS if hasattr(Image, 'Resampling') else Image.LANCZOS
    thumbnail.thumbnail((32, 32), resample)
    canvas = Image.new('RGB', (32, 32), (255, 255, 255))
    offset = ((32 - thumbnail.width) // 2, (32 - thumbnail.height) // 2)
    canvas.paste(thumbnail, offset)
    data = np.asarray(canvas, dtype=np.uint8)
    record = (b'\x01' + data[:, :, 0].tobytes() + data[:, :, 1].tobytes()
              + data[:, :, 2].tobytes())
    assert data.shape == (32, 32, 3) and len(record) == 3073
    record_path.parent.mkdir(parents=True, exist_ok=True)
    record_path.write_bytes(record)
    return rgb, canvas, thumbnail.size, offset

rgb, canvas, thumb_size, offset = prepare_bnn_cifar(IMAGE, CIFAR_FILE)
print('Selected:', IMAGE_INDEX, IMAGE.name)
print('Ground truth:', EXPECTED, CLASS_NAMES[EXPECTED])
print('Original:', rgb.size, '| thumbnail:', thumb_size, '| offset:', offset)
print('CIFAR record:', CIFAR_FILE, '| bytes:', CIFAR_FILE.stat().st_size)
fig, axes = plt.subplots(1, 2, figsize=(6, 3))
axes[0].imshow(rgb)
axes[0].set_title('Original GTSRB')
axes[1].imshow(canvas)
axes[1].set_title('BNN-PYNQ input (32x32)')
for axis in axes:
    axis.axis('off')
plt.tight_layout()
plt.show()


## 6 — Classify the selected image using ARM software

The software-only C++ implementation is loaded into a **separate Python subprocess**. The code below uses the *already prepared* `CIFAR_FILE`, which changes with `IMAGE_INDEX`, and prints the predicted label, expected label, scores, and library-reported inference time. No PL bitstream is loaded.


In [ ]:
import json

CHILD_CODE = """\\
import ctypes, json, sys
from pathlib import Path
so, param, data, nclasses = sys.argv[1:]
lib = ctypes.CDLL(str(Path(so).resolve()))
lib.load_parameters.argtypes = [ctypes.c_char_p]
lib.load_parameters.restype = None
lib.inference.argtypes = [ctypes.c_char_p, ctypes.POINTER(ctypes.c_int), ctypes.c_int,
                          ctypes.POINTER(ctypes.c_float)]
lib.inference.restype = ctypes.c_int
lib.deinit.argtypes = []
lib.deinit.restype = None
lib.load_parameters(str(Path(param).resolve()).encode('utf-8'))
score = (ctypes.c_int * 64)()
us = ctypes.c_float()
pred = lib.inference(str(Path(data).resolve()).encode('utf-8'), score,
                     int(nclasses), ctypes.byref(us))
lib.deinit()
print('BNN_RESULT=' + json.dumps({'predicted':pred, 'microseconds':us.value,
                                  'scores':list(score)[:int(nclasses)]}), flush=True)
"""

if not READY:
    raise RuntimeError('Library or parameter files incomplete. Run section 3.')
if not CIFAR_FILE.is_file() or CIFAR_FILE.stat().st_size != 3073:
    raise FileNotFoundError('CIFAR record missing. Run section 5.')

cmd = [sys.executable, '-c', CHILD_CODE, str(SO_FILE), str(PARAMS),
       str(CIFAR_FILE), str(len(CLASS_NAMES))]
try:
    run = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE,
                         universal_newlines=True, timeout=120)
    print('Exit code:', run.returncode)
    if run.stderr:
        print('STDERR:', run.stderr[-4000:])
    lines = [line for line in run.stdout.splitlines() if line.startswith('BNN_RESULT=')]
    if run.returncode == 0 and lines:
        RESULT = json.loads(lines[-1].split('=', 1)[1])
        pred = int(RESULT['predicted'])
        print('Image:', IMAGE.name)
        print('PREDICTION:', pred, CLASS_NAMES[pred] if 0 <= pred < len(CLASS_NAMES) else '(invalid class)')
        print('EXPECTED:', EXPECTED, CLASS_NAMES[EXPECTED])
        print('MATCH:', pred == EXPECTED)
        print('Inference time:', round(RESULT['microseconds'] / 1e6, 3), 'seconds (library timing)')
        print('Class scores:', RESULT['scores'])
    else:
        print('No valid result. STDOUT:')
        print(run.stdout[-8000:])
        raise RuntimeError('BNN software subprocess failed (exit code {}).'.format(run.returncode))
except subprocess.TimeoutExpired:
    raise TimeoutError('BNN process exceeded 120 seconds.')


## 7 — Test all five images (optional)

This test is independent of `IMAGE_INDEX`. It performs five real predictions **in a single isolated ARM process**, loading the trained parameters just once. It reports one result per photo, the number correct, mean library-reported inference time and throughput. The reported five-image accuracy is **illustrative, not a statistically reliable model accuracy estimate**.

For full GTSRB evaluation we would need many more official test images, including a representative number from each class.


In [ ]:
# Produce the identical 3073-byte CIFAR records for all five sample images.
batch_records = []
for sample in SAMPLES:
    record_path = ASSET_DIR / ('test_' + sample['path'].stem + '.cifar')
    prepare_bnn_cifar(sample['path'], record_path)
    batch_records.append(record_path)

CHILD_BATCH_CODE = """\\
import ctypes, json, sys
from pathlib import Path
so, param, nclasses, *files = sys.argv[1:]
nclasses = int(nclasses)
lib = ctypes.CDLL(str(Path(so).resolve()))
lib.load_parameters.argtypes = [ctypes.c_char_p]
lib.load_parameters.restype = None
lib.inference.argtypes = [ctypes.c_char_p, ctypes.POINTER(ctypes.c_int), ctypes.c_int,
                          ctypes.POINTER(ctypes.c_float)]
lib.inference.restype = ctypes.c_int
lib.deinit.argtypes = []
lib.deinit.restype = None
lib.load_parameters(str(Path(param).resolve()).encode('utf-8'))
results = []
for file in files:
    score = (ctypes.c_int * 64)()
    microseconds = ctypes.c_float()
    prediction = lib.inference(str(Path(file).resolve()).encode('utf-8'),
                               score, nclasses, ctypes.byref(microseconds))
    results.append({'predicted': prediction, 'microseconds': microseconds.value})
lib.deinit()
print('BNN_RESULTS=' + json.dumps(results), flush=True)
"""

if not READY:
    raise RuntimeError('Original BNN library/parameters not ready.')
cmd = ([sys.executable, '-c', CHILD_BATCH_CODE, str(SO_FILE), str(PARAMS),
        str(len(CLASS_NAMES))] + [str(f) for f in batch_records])
run = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE,
                     universal_newlines=True, timeout=180)
lines = [s for s in run.stdout.splitlines() if s.startswith('BNN_RESULTS=')]
if run.returncode != 0 or not lines:
    print('Exit code:', run.returncode)
    print('STDOUT:', run.stdout[-5000:])
    print('STDERR:', run.stderr[-5000:])
    raise RuntimeError('Five-image batch inference failed.')

results = json.loads(lines[-1].split('=', 1)[1])
if len(results) != len(SAMPLES):
    raise ValueError('Wrong number of predictions from BNN subprocess.')
print('{:<4} {:<12} {:<23} {:<23} {:<7} {:>9}'.format(
    '#', 'Image', 'Expected', 'Predicted', 'Correct', 'Time(s)'))
print('-' * 87)
correct = 0
for i, (sample, res) in enumerate(zip(SAMPLES, results)):
    pred = int(res['predicted'])
    expected = sample['class_id']
    hit = (pred == expected)
    correct += int(hit)
    name = CLASS_NAMES[pred] if 0 <= pred < len(CLASS_NAMES) else 'INVALID'
    print('{:<4} {:<12} {:<23} {:<23} {:<7} {:>9.3f}'.format(
        i, sample['name'], CLASS_NAMES[expected][:22], name[:22],
        'YES' if hit else 'NO', res['microseconds'] / 1e6))

seconds = sum(float(r['microseconds']) for r in results) / 1e6
print('\nCorrect: {} / {} (sample accuracy {:.1f}%)'.format(
    correct, len(results), 100.0 * correct / len(results)))
print('Mean library-reported inference: {:.3f} s/image'.format(seconds / len(results)))
print('Throughput (based on library times): {:.3f} images/s'.format(len(results) / seconds))
print('Note: five images are too few to estimate overall GTSRB accuracy.')


## Interpretation

- **Single-image recognition:** change `IMAGE_INDEX` in section 5, run sections 5–6, inspect the prediction and library time. Incorrect classification can be a genuine model error; don't change the ground-truth label.
- **Five-image evaluation:** section 7 provides a small illustrative accuracy and average ARM inference time with original pretrained parameters. It is not a benchmark of all 43 sign classes.
- **Software only:** no FPGA bitstream is loaded. The reported time is the original ARM software library's own inference timing, **not** a PL-kernel latency or PS–PL transfer measurement.
- **FPGA follow-up:** resource reports show the original complete CNV-W1A1 overlay for PYNQ-Z2 does not fit the Zybo Zynq-7010. Hardware acceleration for Lab08.3 will therefore require a smaller or partial accelerator, built for xc7z010.
